# Linux `cpio` Archiving

## Complete Educational Notebook

This notebook explains `cpio` as an alternative Linux archiving method, including archive creation, extraction, compression pipelines, formats, file extensions, comparison with `tar`, practical labs, and review questions.

## Learning objectives

You will learn:

- what `cpio` is,
- copy-out, copy-in, and pass-through modes,
- archive creation and extraction,
- listing archive contents,
- gzip/bzip2/xz pipelines,
- common file extensions,
- whether `cpio` is better than `tar`,
- initramfs and recovery use cases.

## 1. What is `cpio`?

`cpio` is a Unix/Linux archiving utility. Its name is commonly understood as **CoPy In and Out**. It creates, extracts, lists, or directly copies file trees. It is primarily an archiver, not a compressor.

## 2. The three modes

| Mode | Option | Meaning |
|---|---|---|
| Copy-out | `-o` | Create an archive |
| Copy-in | `-i` | Extract or list an archive |
| Pass-through | `-p` | Copy files directly to another directory |

## 3. Why `find` is commonly used

`cpio` commonly reads pathnames from standard input.

```bash
find project -print | cpio -o > project.cpio
```

`find` generates the file list; `cpio` archives those names.

## 4. Creating an archive

```bash
find project -print | cpio -o > project.cpio
```

Recommended modern format:

```bash
find project -print | cpio -o -H newc > project.cpio
```

## 5. Safer filename handling

Use null-delimited names:

```bash
find project -print0 | cpio --null -o -H newc > project.cpio
```

## 6. Verbose creation

```bash
find project -print | cpio -ov -H newc > project.cpio
```

`-v` displays processed paths.

## 7. Listing contents

```bash
cpio -it < project.cpio
```

Verbose listing:

```bash
cpio -itv < project.cpio
```

## 8. Extracting

```bash
cpio -idmv < project.cpio
```

- `-i`: extract
- `-d`: create directories
- `-m`: preserve modification times
- `-v`: verbose

## 9. Overwriting files

```bash
cpio -idu < project.cpio
```

`-u` allows replacement of existing files. Use carefully.

## 10. Extracting selected files

```bash
cpio -id '*.txt' < project.cpio
```

Quote the pattern so the shell does not expand it first.

## 11. Extracting elsewhere

```bash
mkdir restore
cd restore
cpio -idmv < ../project.cpio
```

## 12. `cpio` does not compress automatically

An uncompressed archive typically has extension `.cpio`. Compression is done by piping through another utility.

## 13. gzip-compressed cpio

Create:
```bash
find project -print0 | cpio --null -o -H newc | gzip > project.cpio.gz
```
Extract:
```bash
gzip -dc project.cpio.gz | cpio -idmv
```

## 14. bzip2-compressed cpio

Create:
```bash
find project -print0 | cpio --null -o -H newc | bzip2 > project.cpio.bz2
```
Extract:
```bash
bzip2 -dc project.cpio.bz2 | cpio -idmv
```

## 15. xz-compressed cpio

Create:
```bash
find project -print0 | cpio --null -o -H newc | xz > project.cpio.xz
```
Extract:
```bash
xz -dc project.cpio.xz | cpio -idmv
```

## 16. Pass-through mode

```bash
find project -print | cpio -pdmv backup/
```

- `-p`: pass-through
- `-d`: create directories
- `-m`: preserve timestamps
- `-v`: verbose

## 17. Hard-link pass-through

Some implementations support:
```bash
find project -print | cpio -pdl backup/
```
`-l` attempts to create hard links when possible.

## 18. Archive formats

| Format | Option |
|---|---|
| binary | `-H bin` |
| old ASCII | `-H odc` |
| new ASCII | `-H newc` |
| CRC | `-H crc` |
| tar | `-H tar` |
| ustar | `-H ustar` |

## 19. Why `newc` is useful

`newc` supports modern filenames and metadata ranges and is common in Linux initramfs images.

## 20. Common file extensions

| Type | Extension |
|---|---|
| uncompressed | `.cpio` |
| gzip compressed | `.cpio.gz` |
| bzip2 compressed | `.cpio.bz2` |
| xz compressed | `.cpio.xz` |
| zstd compressed | `.cpio.zst` |

The correct term is usually **extension** or **suffix**, not prefix.

## 21. Detecting actual format

Use:
```bash
file archive.cpio.gz
```
The extension is only a convention.

## 22. Is `cpio` better than `tar`?

Neither is universally better. `tar` is usually simpler for everyday directory archiving. `cpio` is especially useful with `find`, selective archiving, pass-through copying, initramfs, and recovery systems.

## 23. `tar` versus `cpio`

| Feature | `tar` | `cpio` |
|---|---|---|
| Archive a directory directly | Easy | Usually via file list |
| Input style | command arguments | stdin pathname list |
| Compression integration | very convenient | usually explicit pipeline |
| Pass-through copying | not primary | built in |
| Initramfs | uncommon | common |

## 24. Real-world application: initramfs

Linux initramfs images are often compressed `cpio` archives, commonly using `newc`.

## 25. Selective backups

```bash
find /etc -type f -mtime -7 -print0 | cpio --null -o -H newc | gzip > recent-configs.cpio.gz
```

## 26. Important options

| Option | Meaning |
|---|---|
| `-o` | create archive |
| `-i` | extract/list |
| `-p` | pass-through |
| `-t` | list contents |
| `-v` | verbose |
| `-d` | create directories |
| `-m` | preserve modification time |
| `-u` | overwrite existing |
| `-H` | select archive format |
| `--null` | read null-delimited names |
| `-F FILE` | use archive file |

## 27. Using `-F`

Create:
```bash
find project -print | cpio -o -H newc -F project.cpio
```
Extract:
```bash
cpio -idmv -F project.cpio
```

## 28. Common mistakes

- Expecting `cpio` to scan directories automatically.
- Forgetting `-d` during extraction.
- Confusing archiving with compression.
- Using newline-delimited names for unusual filenames.
- Extracting untrusted archives carelessly.

## 29. Security note

Inspect untrusted archives first:
```bash
cpio -itv < archive.cpio
```

## 30. Review questions

1. What does `cpio` do?
2. Is it an archiver or compressor?
3. What do `-o`, `-i`, and `-p` mean?
4. Why is `find` commonly used?
5. How do you create `.cpio.gz`?
6. How do you extract `.cpio.xz`?
7. What does `-H newc` mean?
8. What are common extensions?
9. Is `cpio` better than `tar`?
10. Why is `cpio` used in initramfs?

## 31. Answers

1. It creates, extracts, lists, or directly copies file trees.
2. Primarily an archiver.
3. Copy-out, copy-in, and pass-through.
4. It generates the pathname list.
5. Pipe `cpio` output through `gzip`.
6. `xz -dc archive.cpio.xz | cpio -idmv`.
7. Select the new ASCII archive format.
8. `.cpio`, `.cpio.gz`, `.cpio.bz2`, `.cpio.xz`.
9. Not universally.
10. Its archive format works well for early userspace filesystem images.

# Hands-On Lab

In [ ]:
rm -rf cpio_lab
mkdir -p cpio_lab/project/src cpio_lab/project/data

printf '# CPIO Lab\n' > cpio_lab/project/README.md
printf 'int main(void){return 0;}\n' > cpio_lab/project/src/main.c
printf 'name,value\nalpha,1\nbeta,2\n' > cpio_lab/project/data/sample.csv

cd cpio_lab
find project -print

In [ ]:
find project -print0 | cpio --null -o -H newc > project.cpio
ls -lh project.cpio
file project.cpio

In [ ]:
cpio -itv < project.cpio

In [ ]:
mkdir restore
cd restore
cpio -idmv < ../project.cpio
cd ..
find restore -type f -print

In [ ]:
sha256sum project/README.md restore/project/README.md
sha256sum project/src/main.c restore/project/src/main.c
sha256sum project/data/sample.csv restore/project/data/sample.csv

In [ ]:
find project -print0 | cpio --null -o -H newc | gzip > project.cpio.gz
gzip -t project.cpio.gz
gzip -dc project.cpio.gz | cpio -itv

In [ ]:
find project -print0 | cpio --null -o -H newc | bzip2 > project.cpio.bz2
find project -print0 | cpio --null -o -H newc | xz > project.cpio.xz
ls -lh project.cpio*

In [ ]:
mkdir pass-copy
find project -print0 | cpio --null -pdmv pass-copy
find pass-copy -type f -print

# Cheat Sheet

```bash
# Create
find project -print | cpio -o -H newc > project.cpio

# List
cpio -itv < project.cpio

# Extract
cpio -idmv < project.cpio

# gzip
find project -print0 | cpio --null -o -H newc | gzip > project.cpio.gz
gzip -dc project.cpio.gz | cpio -idmv

# bzip2
find project -print0 | cpio --null -o -H newc | bzip2 > project.cpio.bz2
bzip2 -dc project.cpio.bz2 | cpio -idmv

# xz
find project -print0 | cpio --null -o -H newc | xz > project.cpio.xz
xz -dc project.cpio.xz | cpio -idmv

# Pass-through
find project -print0 | cpio --null -pdmv destination/
```